# Traffic Sign Robustness Assessment - Starter

Complete this notebook to evaluate a traffic sign recognition model under clean, environmental, and adversarial conditions. Your final output should include a CSV scorecard, visual comparisons, a metric comparison chart, and a short operational assessment report.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))
os.environ["ART_DATA_PATH"] = str(ROOT / "art_data")
os.environ["USERPROFILE"] = str(ROOT)
os.environ["HOME"] = str(ROOT)
os.environ["MPLCONFIGDIR"] = str(ROOT / ".matplotlib")

import torch
from traffic_sign_robustness_utils import (
    calculate_condition_metrics,
    environmental_test_sets,
    load_subset,
    perturbation_linf,
    plot_clean_vs_degraded,
    plot_metric_bars,
    prepare_gtsrb_subsets,
    row_from_metrics,
    run_fgsm_attacks,
    train_or_load_model,
    write_assessment_report,
    write_scorecard,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
device

## 1. Load the Traffic Sign Dataset and Model

The infrastructure code creates a compact six-class GTSRB subset. The first run trains a small classroom model and saves the checkpoint.

In [ ]:
data_dir = ROOT / "data" / "generated"
download_dir = ROOT / "data" / "gtsrb"
model_path = ROOT / "models" / "traffic_sign_cnn.pt"
results_dir = ROOT / "results"

prepare_gtsrb_subsets(data_dir, download_dir, train_per_class=80, val_per_class=20)
train_images, train_labels = load_subset(data_dir, "train")
val_images, val_labels = load_subset(data_dir, "val")

model = train_or_load_model(model_path, train_images, train_labels, device=device, epochs=12)
val_images.shape, val_labels.shape

## 2. Measure Clean Baseline Metrics

Complete `calculate_condition_metrics` in `src/traffic_sign_robustness_utils.py`, then run this cell.

In [ ]:
# TODO: Implement calculate_condition_metrics before running this cell.
clean_metrics = calculate_condition_metrics(model, val_images, val_labels, device=device)
clean_predictions = clean_metrics["predictions"]
clean_confidence = clean_metrics["confidence"]

rows = [row_from_metrics("clean", "baseline", clean_metrics, perturbation=0.0)]
rows

## 3. Evaluate Environmental Perturbations

Complete the Gaussian noise and environmental test-set functions. Your workflow must include blur, noise, and compression artifacts.

In [ ]:
# TODO: Implement gaussian_noise, environmental_test_sets, and perturbation_linf.
first_environmental_set = None

for condition, degraded_images in environmental_test_sets(val_images).items():
    metrics = calculate_condition_metrics(
        model,
        degraded_images,
        val_labels,
        clean_predictions=clean_predictions,
        clean_confidence=clean_confidence,
        device=device,
    )
    rows.append(row_from_metrics(condition, "environmental", metrics, perturbation_linf(val_images, degraded_images)))
    if first_environmental_set is None:
        first_environmental_set = (condition, degraded_images)

[row for row in rows if row["condition_type"] == "environmental"]

## 4. Run ART Adversarial Perturbations

Complete `run_fgsm_attacks` and test at least three perturbation strengths.

In [ ]:
# TODO: Implement run_fgsm_attacks with ART FastGradientMethod.
first_adversarial_set = None

for condition, adversarial_images in run_fgsm_attacks(model, val_images, epsilons=[0.01, 0.03, 0.06]).items():
    metrics = calculate_condition_metrics(
        model,
        adversarial_images,
        val_labels,
        clean_predictions=clean_predictions,
        clean_confidence=clean_confidence,
        device=device,
    )
    rows.append(row_from_metrics(condition, "adversarial", metrics, perturbation_linf(val_images, adversarial_images)))
    if first_adversarial_set is None:
        first_adversarial_set = (condition, adversarial_images)

[row for row in rows if row["condition_type"] == "adversarial"]

## 5. Generate Required Outputs

Write the CSV scorecard, metric chart, visual comparisons, and Markdown report.

In [ ]:
# TODO: Complete write_assessment_report before running this cell.
scorecard_path = write_scorecard(rows, results_dir / "traffic_sign_robustness_scorecard.csv")
chart_path = plot_metric_bars(rows, results_dir / "traffic_sign_metric_comparison.png")
report_path = write_assessment_report(rows, results_dir / "traffic_sign_assessment_report.md")

if first_environmental_set is not None:
    condition, degraded_images = first_environmental_set
    plot_clean_vs_degraded(val_images, degraded_images, results_dir / "sample_environmental_degradation.png", f"Clean vs {condition}")

if first_adversarial_set is not None:
    condition, adversarial_images = first_adversarial_set
    plot_clean_vs_degraded(val_images, adversarial_images, results_dir / "sample_adversarial_degradation.png", f"Clean vs {condition}")

scorecard_path, chart_path, report_path

In [ ]:
sorted(rows, key=lambda row: (row["risk_level"], row["accuracy"]))